# Analysis of Instagram DMs through Zipf's law

## Useful imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm 
import json
from pathlib import Path
import nltk
from nltk.corpus import stopwords

import re
from collections import Counter, defaultdict
import pandas as pd
import seaborn as sns

In [ ]:
#nltk.download('punkt_tab')
#nltk.download("stopwords")
stopwords_fr = set(stopwords.words("french"))
stopwords_en = set(stopwords.words("english"))
STOPWORDS_USED = False

## Import of the corpus

In [ ]:
json_raw_dict = {}

for file in Path("corpus").glob("*.json"):
    user = file.stem

    with open(file, "r", encoding="utf-8") as f:
        json_raw_dict[user] = json.load(f)

# A bit of error handling
if not json_raw_dict:
    raise FileNotFoundError("Corpus vide !")

print(f"{len(json_raw_dict)} conversations importées avec succès !")

In [ ]:
json_raw_dict["235165458121288"]["participants"]

In [ ]:
def fix_encoding(text):
    return text.encode("latin1").decode("utf-8")

## Retrieval and reordering of the messages 

In [ ]:
def filter_content(message):
    is_attachment = "envoy\u00c3\u00a9 une pi\u00c3\u00a8ce jointe" in message
    is_reaction = re.fullmatch(
		r"Reacted .+ to your message ",
		message
	)
    is_like = message == "Liked a message"
    is_video_chat_pic = re.fullmatch(
        r".+ took a photo of the video chat",
        message
	)
    is_video_chat_end = message == "Video chat ended"
    return is_attachment or is_reaction or is_like or is_video_chat_pic or is_video_chat_end

In [ ]:
for user in json_raw_dict:
    for individual_message in reversed(json_raw_dict[user]["messages"]):
        #if "content" in individual_message.keys() and (individual_message["content"] in ["Vous avez envoy\u00c3\u00a9 une pi\u00c3\u00a8ce jointe.", "Liked a message"]):
        if "content" in individual_message.keys() and filter_content(individual_message["content"]):
            pass
			#print(individual_message)
        # try:
        #     print(individual_message["content"])
        # except KeyError:
        #     print("woops")

In [ ]:
json_messages_dict = {
    user: {
        "participants": [{'name': fix_encoding(participant["name"])} for participant in json_raw_dict[user]["participants"]],
        "messages": [
            (fix_encoding(individual_message["sender_name"]), fix_encoding(individual_message["content"]))
        	for individual_message in reversed(json_raw_dict[user]["messages"]) 
            if "content" in individual_message
            and not filter_content(individual_message["content"])
		]
    } 
    for user in json_raw_dict
}

In [ ]:
json_messages_dict

In [ ]:
EXCLUDED_USERS = {"Utilisateur Instagram"}

for conversation in json_messages_dict.values():
    conversation["messages"] = [
        (sender, content)
        for sender, content in conversation["messages"]
        if sender not in EXCLUDED_USERS
    ]

    conversation["participants"] = [
        p for p in conversation["participants"]
        if p["name"] not in EXCLUDED_USERS
    ]

for user in json_messages_dict: 
    # Ajouter les mots pour obtenir la fréquence par user et par conv
    #print(json_messages_dict[user]["participants"])
    word_list = {participant["name"]: [] for participant in json_messages_dict[user]["participants"]}
    for message in json_messages_dict[user]["messages"]:
        try:
            word_list[message[0]] += nltk.word_tokenize(message[1].lower())
        except KeyError:
            # print(f"Participants : {json_messages_dict[user]["participants"]}")
            # print(f"Message : {message}")
            pass
    #print(word_list)
    word_count_dict = {
        participant: {
        	word: word_list[participant].count(word) 
        	for word in word_list[participant]
    	}
        for participant in word_list
    }
    print(word_count_dict)
    json_messages_dict[user]["words"] = word_count_dict

In [ ]:
# Compter en O(nombre de tokens), plutôt qu'avec list.count() à répétition.
# Les lettres accentuées et les apostrophes internes sont conservées.
TOKEN_RE = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*", re.UNICODE)

for user, conversation in tqdm(json_messages_dict.items(), desc="Tokenisation"):
    counts_by_sender = {
        participant["name"]: Counter()
        for participant in conversation["participants"]
        if "name" in participant
    }
    for sender, content in conversation["messages"]:
        if not isinstance(content, str):
            continue
        #tokens = TOKEN_RE.findall(content.casefold())
        tokens = [
            word
			for word in TOKEN_RE.findall(content.casefold())
            if word not in stopwords_fr
        ] if STOPWORDS_USED else TOKEN_RE.findall(content.casefold())
        counts_by_sender.setdefault(sender, Counter()).update(tokens)

    conversation["words"] = counts_by_sender

print("Fréquences calculées pour chaque participant, dans chaque conversation.")

for user in json_messages_dict:
    for participant in json_messages_dict[user]["words"]:
        word_dict = json_messages_dict[user]["words"][participant]
        plt.hist(word_dict.keys(), word_dict.values())
        plt.show()

## Most frequent words per participant per conversation

In [ ]:
def plot_top_words(conversation_id, participant, n=25):
    counts = json_messages_dict[conversation_id]["words"][participant]
    most_common = counts.most_common(n)
    if not most_common:
        print(f"Aucun mot pour {participant!r} dans {conversation_id!r}.")
        return

    words, frequencies = zip(*most_common)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(words, frequencies)
    if participant != "Alexandre Pts":
        print(f"PARTICIPANT IS {participant}")
        participant_name = "Anonymous"
    else:
        participant_name = participant
    ax.set(title=f"Top {len(words)} words — {participant_name}",
           xlabel="Word", ylabel="Number of occurrences")
    plt.setp(ax.get_xticklabels(), rotation=50, ha="right")
    fig.tight_layout()
    plt.show()

## Plots to check Zipf's law

In [ ]:
def plot_zipf(conversation_id, participants=None, min_count=1, max_rank=None):
    by_sender = json_messages_dict[conversation_id]["words"]
    if participants is None:
        participants = list(by_sender)

    fig, ax = plt.subplots(figsize=(10, 6))
    reference_added = False
    for participant in participants:
        if participant != "Alexandre Pts":
            print(f"PARTICIPANT IS {participant}")
            participant_name = "Anonymous"
        else:
            participant_name = participant
        counts = by_sender[participant]
        ordered_counts = np.array(
            [n for _, n in counts.most_common() if n >= min_count],
            dtype=float
        )
        if max_rank is not None:
            ordered_counts = ordered_counts[:max_rank]
        if not len(ordered_counts):
            continue

        ranks = np.arange(1, len(ordered_counts) + 1)
        frequencies = ordered_counts / counts.total()
        ax.loglog(ranks, frequencies, '.', markersize=4, label=participant_name)

        if not reference_added:
            ax.loglog(ranks, frequencies[0] / ranks, '--',
                      linewidth=1, color='gray', label='Reference 1/r')
            ax.loglog(ranks, frequencies[len(frequencies)//2]*ranks[len(frequencies)//2] / ranks, '--',
						linewidth=1, color='red', label='Reference 1/r with offset')
            reference_added = True

    if conversation_id == "marie_1096274075343914":
        ax.set(xlabel="Word's rank (log)", ylabel="Relative frequency (log)",
			title=f"Zipf — conversation with a close friend")
    else:
        ax.set(xlabel="Word's rank (log)", ylabel="Relative frequency (log)",
					title=f"Zipf — conversation with a remote friend")
    ax.grid(True, which="both", alpha=.25)
    ax.legend()
    fig.tight_layout()
    plt.show()

## Auto example

In [ ]:
conversation_id = max(
    json_messages_dict,
    key=lambda cid: sum(c.total() for c in json_messages_dict[cid]["words"].values())
)
participant = max(
    json_messages_dict[conversation_id]["words"],
    key=lambda name: json_messages_dict[conversation_id]["words"][name].total()
)
#participant = list(json_messages_dict[conversation_id]["words"].keys())[0]
print(f"Conversation with a close friend")
plot_top_words(conversation_id, participant, n=25)
plot_zipf(conversation_id, min_count=2)  # min_count=1 pour la courbe complète

In [ ]:
conversation_id = "235165458121288"
participant = max(
    json_messages_dict[conversation_id]["words"],
    key=lambda name: json_messages_dict[conversation_id]["words"][name].total()
)
#participant = list(json_messages_dict[conversation_id]["words"].keys())[0]
print(f"Conversation : {conversation_id} | Participant : {participant}")
plot_top_words(conversation_id, participant, n=25)
plot_zipf(conversation_id, min_count=2)  # min_count=1 pour la courbe complète

## Computing of the dataframe

In [ ]:
rows = []
for conversation_id, conversation in json_messages_dict.items():
    for participant, counts in conversation["words"].items():
        total = counts.total()
        for rank, (word, count) in enumerate(counts.most_common(), start=1):
            rows.append({
                "conversation": conversation_id,
                "participant": participant,
                "word": word,
                "count": count,
                "rank": rank,
                "frequency": count / total,
            })

words_df = pd.DataFrame(rows, columns=[
    "conversation", "participant", "word", "count", "rank", "frequency"
])
words_df.head()
filename_addendum_for_csv = "" if STOPWORDS_USED else "no_"
words_df.to_csv(f"word_frequencies_with_{filename_addendum_for_csv}stopwords_filter.csv")

## Aggregation

In [ ]:
global_word_counts = defaultdict(Counter)
for conversation in json_messages_dict.values():
    for participant, counts in conversation["words"].items():
        global_word_counts[participant].update(counts)

# Exemple : mots les plus fréquents de la personne ayant envoyé le plus de mots
most_active = max(global_word_counts, key=lambda name: global_word_counts[name].total())
print(most_active, global_word_counts[most_active].most_common(20))

def plot_global_zipf(participants, min_count=1):
    fig, ax = plt.subplots(figsize=(10, 6))
    for participant in participants:
        counts = global_word_counts[participant]
        ordered = np.array([n for _, n in counts.most_common() if n >= min_count])
        if len(ordered) == 0:
            continue
        ranks = np.arange(1, len(ordered) + 1)
        ax.loglog(ranks, ordered / counts.total(), '.', markersize=4, label=participant)
        ax.loglog(ranks, (ordered[0]/counts.total()) / ranks, '--',
			linewidth=1, color='gray', label='Reference 1/r')
        ax.loglog(ranks, (ordered[int(np.sqrt(len(ordered)))]/counts.total())*ranks[int(np.sqrt(len(ordered)))] / ranks, '--',
			linewidth=1, color='red', label='Reference 1/r with offset (sqrt(2) instead of .5)')
    ax.set(xlabel="Rang (log)", ylabel="Relative frequency (log)",
           title="Zipf — aggregation of all conversations")
    ax.grid(True, which="both", alpha=.25)
    ax.legend()
    plt.show()

plot_global_zipf([most_active], min_count=2)

In [ ]:
TOP_K = 500

# Regrouper les occurrences d'un même mot par participant
df = (
    words_df.groupby(["participant", "word"], as_index=False)
    ["count"].sum()
)

# Calculer les fréquences relatives
df["frequency"] = (
    df["count"] /
    df.groupby("participant")["count"].transform("sum")
)

# Trier les mots par fréquence décroissante
df = df.sort_values(
    ["participant", "frequency"],
    ascending=[True, False]
)

# Attribuer un rang à chaque mot
df["rank"] = df.groupby("participant").cumcount() + 1

# Construire la matrice des TOP_K fréquences
X = (
    df[df["rank"] <= TOP_K]
    .pivot(index="participant", columns="rank", values="frequency")
    .reindex(columns=range(1, TOP_K + 1))
    .fillna(0)
)

X.columns = [f"freq_{i}" for i in X.columns]

# Compter les messages de chaque participant
message_counts = Counter(
    sender
    for conversation in json_messages_dict.values()
    for sender, content in conversation["messages"]
)

# Sélectionner les 100 participants les plus actifs
top_100_users = [user for user, count in message_counts.most_common(50)]

# Filtrer la matrice de fréquences
X = X.loc[X.index.intersection(top_100_users)]

In [ ]:
X.to_csv("matrice_par_user.csv")

In [ ]:
import plotly.express as px

fig = px.imshow(
    np.log10(X.where(X > 0)),
    labels={
        "x": "Word's rank",
        "y": "Participant",
        "color": "log10(frequency)"
    },
    aspect="auto",
	# [
    #     'aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
	# 	'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg',
	# 	'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl',
	# 	'darkmint', 'deep', 'delta', 'dense', 'earth', 'edge', 'electric',
	# 	'emrld', 'fall', 'geyser', 'gnbu', 'gray', 'greens', 'greys',
	# 	'haline', 'hot', 'hsv', 'ice', 'icefire', 'inferno', 'jet',
	# 	'magenta', 'magma', 'matter', 'mint', 'mrybm', 'mygbm', 'oranges',
	# 	'orrd', 'oryel', 'oxy', 'peach', 'phase', 'picnic', 'pinkyl',
	# 	'piyg', 'plasma', 'plotly3', 'portland', 'prgn', 'pubu', 'pubugn',
	# 	'puor', 'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu',
	# 	'rdgy', 'rdpu', 'rdylbu', 'rdylgn', 'redor', 'reds', 'solar',
	# 	'spectral', 'speed', 'sunset', 'sunsetdark', 'teal', 'tealgrn',
	# 	'tealrose', 'tempo', 'temps', 'thermal', 'tropic', 'turbid',
	# 	'turbo', 'twilight', 'viridis', 'ylgn', 'ylgnbu', 'ylorbr',
	# 	'ylorrd'
    # ]
    color_continuous_scale="turbo",
    title="Distribution of lexical frequencies"
)

fig.update_layout(
    width=1200,
    height=800
)

fig.show()

In [ ]:
df

In [ ]:
CHUNK_SIZE = 150     # Mots par exemple : essayer 100, 150, 300...
MIN_CHUNKS = 5       # Nombre minimal d'exemples pour conserver un participant
MAX_USERS = 30       # Les 30 participants les plus actifs (modifier si besoin)

chunk_rows = []
for conversation_id, conversation in json_messages_dict.items():
    buffers = defaultdict(list)
    for participant, content in conversation["messages"]:
        if not isinstance(content, str):
            continue
        tokens = TOKEN_RE.findall(content.casefold())
        if STOPWORDS_USED:
            tokens = [word for word in tokens if word not in stopwords_fr]
        buffers[participant].extend(tokens)

        # Blocs disjoints, sans jamais mélanger les participants ni les conversations
        while len(buffers[participant]) >= CHUNK_SIZE:
            chunk = buffers[participant][:CHUNK_SIZE]
            del buffers[participant][:CHUNK_SIZE]
            chunk_rows.append({
                "participant": participant,
                "conversation": conversation_id,
                "text": " ".join(chunk),
            })

chunks_df = pd.DataFrame(chunk_rows, columns=["participant", "conversation", "text"])
activity = chunks_df["participant"].value_counts()
selected_users = activity[activity >= MIN_CHUNKS].head(MAX_USERS).index
samples_df = chunks_df[chunks_df["participant"].isin(selected_users)].copy()

print(f"{len(samples_df)} blocs pour {len(selected_users)} participants")
display(activity.head(15).rename("nombre_de_blocs").to_frame())

In [ ]:
# Garder approximativement les 80 % premiers blocs pour l'entraînement,
# et les 20 % derniers pour le test, séparément pour chaque participant.
# Ceci conserve les mêmes classes dans train et test sans mélanger les blocs.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
import xgboost as xgb

if samples_df["participant"].nunique() < 2:
    print("Pas assez de participants : réduire MIN_CHUNKS / CHUNK_SIZE ou augmenter MAX_USERS.")
else:
    position = samples_df.groupby("participant").cumcount()
    size = samples_df.groupby("participant")["participant"].transform("size")
    train_mask = position < (0.8 * size).astype(int)

    train_df = samples_df.loc[train_mask]
    test_df = samples_df.loc[~train_mask]

    classifier = make_pipeline(
        TfidfVectorizer(
            tokenizer=str.split,  # les textes sont déjà tokenisés et joints par espaces
            token_pattern=None,
            lowercase=False,
            sublinear_tf=True,
            min_df=2,
            max_features=20000,
        ),
        LinearSVC(class_weight="balanced"),
    )
    classifier.fit(train_df["text"], train_df["participant"])
    y_pred = classifier.predict(test_df["text"])

    print("Accuracy :", round(accuracy_score(test_df["participant"], y_pred), 3))
    print("Balanced accuracy :", round(balanced_accuracy_score(test_df["participant"], y_pred), 3))
    print(classification_report(test_df["participant"], y_pred, zero_division=0))

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Réutiliser les mêmes jeux train/test
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    sublinear_tf=True,
    min_df=2,
    max_features=5000
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

# Encoder les participants en entiers
encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df["participant"])
y_test = encoder.transform(test_df["participant"])

# Définir le modèle
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1
)

# Entraînement
xgb_model.fit(X_train, y_train)

# Prédictions
y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_xgb))

print(classification_report(
    encoder.inverse_transform(y_test),
    encoder.inverse_transform(y_pred_xgb),
    zero_division=0
))

In [ ]:
from collections import Counter

TOP_K_ZIPF = 200

def zipf_features(text, top_k=TOP_K_ZIPF):
    words = text.split()
    counts = Counter(words)

    freqs = np.array(
        sorted(counts.values(), reverse=True)[:top_k],
        dtype=np.float32
    )

    freqs /= max(len(words), 1)

    return np.pad(
        freqs,
        (0, top_k - len(freqs))
    )

X_train_zipf = np.vstack([
    zipf_features(text)
    for text in train_df["text"]
])

X_test_zipf = np.vstack([
    zipf_features(text)
    for text in test_df["text"]
])

In [ ]:
from sklearn.base import clone

xgb_zipf = clone(xgb_model)

xgb_zipf.fit(X_train_zipf, y_train)

y_pred_zipf = xgb_zipf.predict(X_test_zipf)

print("Zipf accuracy:", accuracy_score(y_test, y_pred_zipf))
print("Zipf balanced accuracy:",
      balanced_accuracy_score(y_test, y_pred_zipf))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "LinearSVC + TF-IDF",
        "XGBoost + TF-IDF",
        "XGBoost + Zipf"
    ],
    "Accuracy": [
        accuracy_score(test_df["participant"], y_pred),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_zipf)
    ],
    "Balanced accuracy": [
        balanced_accuracy_score(test_df["participant"], y_pred),
        balanced_accuracy_score(y_test, y_pred_xgb),
        balanced_accuracy_score(y_test, y_pred_zipf)
    ]
})

display(results)

In [ ]:
def count_words(text, remove_stopwords=STOPWORDS_USED):
    tokens = TOKEN_RE.findall(text.casefold())

    if remove_stopwords:
        tokens = [word for word in tokens if word not in stopwords_en]

    return Counter(tokens)

In [ ]:
def plot_top_words_text(text, n=25, title="Document"):
    counts = count_words(text)
    most_common = counts.most_common(n)

    if not most_common:
        return

    words, frequencies = zip(*most_common)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(words, frequencies)

    ax.set(
        title=f"Top {len(words)} words — {title}",
        xlabel="Word",
        ylabel="Number of occurrences"
    )

    plt.setp(ax.get_xticklabels(), rotation=50, ha="right")
    fig.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
def plot_zipf_text(text, title="Document", min_count=1, max_rank=None):
    counts = count_words(text)

    ordered_counts = np.array(
        [n for _, n in counts.most_common() if n >= min_count],
        dtype=float
    )

    if max_rank is not None:
        ordered_counts = ordered_counts[:max_rank]

    if len(ordered_counts) == 0:
        return

    ranks = np.arange(1, len(ordered_counts) + 1)
    frequencies = ordered_counts / counts.total()

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.loglog(
        ranks, frequencies,
        '.', markersize=4,
        label=title
    )

    ax.loglog(
        ranks, frequencies[0] / ranks,
        '--', linewidth=1,
        color="gray",
        label="Reference 1/r"
    )

    ax.loglog(
		ranks, frequencies[int(np.sqrt(len(frequencies)))]*ranks[int(np.sqrt(len(frequencies)))] / ranks,
		'--', linewidth=1,
		color="gray",
		label="Reference 1/r with offset (sqrt(2) instead of .5)"
	)

    ax.set(
        xlabel="Word's rank (log)",
        ylabel="Relative frequency (log)",
        title=f"Zipf — {title}"
    )

    ax.grid(True, which="both", alpha=.25)
    ax.legend()
    fig.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
from pathlib import Path
from pylatexenc.latex2text import LatexNodes2Text

latex_source = Path("/home/aprats/Documents/Akheros/Schiebel_stage/Rapport-Stage-3A/main.tex").read_text(encoding="utf-8")

text = LatexNodes2Text().latex_to_text(latex_source)

plot_top_words_text(text, n=30, title="Internship report in English")
plot_zipf_text(text, title="Internship report in English", min_count=2)